In [8]:
import polars as pl
import numpy as np
import xarray as xr 
import dask

exp_id_dict = {
    'historical' : 'SPEAR_c192_o1_Hist_AllForc_IC1921_K50',
    'scenarioSSP5-85' : 'SPEAR_c192_o1_Scen_SSP585_IC2011_K50'
}

table_dict = {
    'psl' : {'6hr' : 'atmos_4xdaily'}
}

var_names = {
    'psl' : 'slp'
}

columns = ['min', 'max', 'mean', 'std']

def get_member_id(variant_label):
    return "pp_ens_" + variant_label.split("i")[0][1:].zfill(2)

cmor_out_catalog = "/work/a3r/catalogs/cmip6_psl_catalog/catalog.csv"
csv_base_path = "/work/a3r/Documents/catalog_thumbs/tftest/"

cat_df = pl.read_csv(cmor_out_catalog)

# use cmorized values
var_id = "psl"
exp_id = "historical"
table_id = "6hr"

In [2]:
subcat = cat_df.filter(
    (pl.col("variable_id") == var_id) &
    (pl.col("experiment_id") == exp_id) &
    (pl.col("table_id") == table_id)
)

correct_values = pl.read_csv(
    csv_base_path + 
    exp_id_dict[exp_id] + '/' +
    table_dict[var_id][table_id] + '/' +
    var_names[var_id] + '.csv'
)

In [17]:
are_values_correct = []

In [24]:
variant_id = 'r1i1p1f1'

df = subcat.filter(pl.col('member_id') == variant_id)

# var_id = df['variable_id'].unique()[0]
# if len(df['path']) == 1:
ds = xr.open_dataset(
    df['path'][0], 
    chunks='auto',
    engine='netcdf4'
)[var_id].isel(time=0)
# else:
#     ds = xr.open_mfdataset(
#         df['path'], 
#         parallel=True, 
#         chunks='auto',
#         engine='netcdf4'
#     )[var_id]

weights = np.cos(np.deg2rad(ds.lat))
weights.name = "weights"

In [5]:
avgval_lazy = ds.weighted(weights).mean()
stdval_lazy = ds.weighted(weights).std()

minval_lazy = ds.min()
maxval_lazy = ds.max()

cmor_values = np.array(dask.compute(
    minval_lazy,
    maxval_lazy,
    avgval_lazy,
    stdval_lazy
))

In [19]:
expected_values = correct_values.filter(
    pl.col('member_id') == get_member_id(variant_id)
).select(pl.exclude('member_id')).to_numpy()

res = {i : j for i,j in zip(columns, (cmor_values == expected_values)[0])}
res['member_id'] = variant_id

are_values_correct.append(res)

are_values_correct

[{'min': np.False_,
  'max': np.False_,
  'mean': np.False_,
  'std': np.False_,
  'member_id': 'r1i1p1f1'},
 {'min': np.False_,
  'max': np.False_,
  'mean': np.False_,
  'std': np.False_,
  'member_id': 'r1i1p1f1'}]

In [23]:
pl.DataFrame(are_values_correct, {'member_id' : str, 'min' : bool, 'max' : bool, 'mean' : bool, 'std': bool}).select(correct_values.columns)

member_id,min,max,mean,std
str,bool,bool,bool,bool
"""r1i1p1f1""",false,false,false,false
"""r1i1p1f1""",false,false,false,false


In [12]:
(cmor_values == expected_values)[0]

array([False, False, False, False])

In [ ]:

res

{'min': np.False_,
 'max': np.False_,
 'mean': np.False_,
 'std': np.False_,
 'member_id': 'r1i1p1f1'}

In [11]:
expected_values[['min', 'max', 'mean', 'std']]

min,max,mean,std
f64,f64,f64,f64
89035.6875,108078.851562,101037.208619,1190.875545


In [14]:
obs_vals = pl.DataFrame({i : j for i,j in zip(['min', 'max', 'mean', 'std'], cmor_values)})

In [32]:
np.array(cmor_values) == expected_values.select(pl.exclude('member_id')).to_numpy()

array([[False, False, False, False]])

In [24]:
expected_values.select(pl.exclude('member_id'))

min,max,mean,std
f64,f64,f64,f64
89035.6875,108078.851562,101037.208619,1190.875545


In [30]:
(obs_vals == expected_values[['min', 'max', 'mean', 'std']]).to_numpy().all()

np.False_